In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import textwrap
from tabulate import tabulate
import json

# Download resource tokenizer
nltk.download('punkt')
nltk.download('punkt_tab')

# Cek GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Hardware yang digunakan: {device}")

if device.type == 'cuda':
    print(f"🚀 GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"   Memory Usage: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
else:
    print("⚠️ WARNING: GPU tidak terdeteksi. Proses IndoT5 akan sangat lambat di CPU.")

d:\conda_envs\nlp_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\conda_envs\nlp_project\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


✅ Hardware yang digunakan: cuda
🚀 GPU Detected: NVIDIA GeForce RTX 3060 Laptop GPU
   Memory Usage: 0.00 GB


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\andik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\andik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# =========================================================
# LOAD DATA
# =========================================================

filename = "tempo_preprocessed_tabulated_fix.csv"

df = pd.read_csv(filename)

# Kolom title tidak dipakai
df = df.drop(columns=["Title", "Source_Sheet"], errors="ignore")

# Pastikan kolom utama ada dan tidak null
df = df.dropna(subset=["step1_cleaned"]).reset_index(drop=True)

print(f"📂 Data Siap: {len(df)} dokumen")
print("Kolom:", df.columns.tolist())
display(df.head(2))

📂 Data Siap: 300 dokumen
Kolom: ['No', 'Content', 'step1_cleaned', 'final_result']


,No,Content,step1_cleaned,final_result
0,1,Indeks Harga Saham Gabungan (IHSG) Bursa Efek ...,Indeks Harga Saham Gabungan (IHSG) Bursa Efek ...,indeks harga saham gabung ihsg bursa efek indo...
1,2,PT Daya Intiguna Yasa Tbk. atau MR DIY Indones...,PT Daya Intiguna Yasa Tbk. atau MR DIY Indones...,pt daya intiguna yasa tbk mr diy indonesia put...


In [3]:
MODEL_NAME = "Wikidepia/IndoT5-base-paraphrase"

print("🔄 Loading IndoT5 Model ke GPU...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Pindahkan model ke GPU
model = model.to(device)
model.eval() # Set ke mode evaluasi (bukan training)

print("✅ Model IndoT5 siap di GPU!")

🔄 Loading IndoT5 Model ke GPU...


d:\conda_envs\nlp_project\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W0518 22:59:10.739173 5076 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
d:\conda_envs\nlp_project\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


✅ Model IndoT5 siap di GPU!


In [4]:
def hybrid_summarization(text, top_n=3):
    """
    Input: Teks bersih (tapi masih ada tanda baca untuk split kalimat)
    Output: (Extractive Summary, Abstractive Summary)
    """
    # --- 1. TEXTRANK (EXTRACTIVE) ---
    sentences = sent_tokenize(text)
    
    # Validasi: Jika kalimat terlalu sedikit, tidak perlu diringkas
    if len(sentences) <= 1:
        return text, text

    # Vectorization & Similarity
    # Menggunakan TF-IDF pada level kalimat
    vectorizer = TfidfVectorizer()
    try:
        tfidf_matrix = vectorizer.fit_transform(sentences)
        similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
    except ValueError:
        # Fallback jika dokumen kosong/stopword semua
        return text, text

    # Graph Construction
    nx_graph = nx.from_numpy_array(similarity_matrix)
    
    # PageRank Algorithm
    try:
        scores = nx.pagerank(nx_graph, max_iter=500) # Tambah iterasi biar konvergen
    except nx.PowerIterationFailedConvergence:
        scores = {i: 0 for i in range(len(sentences))}

    # Ranking & Selection
    ranked_sentences = sorted(((scores[i], s, i) for i, s in enumerate(sentences)), reverse=True)
    
    # Ambil Top-N dan urutkan kembali berdasarkan posisi asli di teks
    selected_sentences = sorted(ranked_sentences[:top_n], key=lambda x: x[2])
    extractive_summary = " ".join([s[1] for s in selected_sentences])
    
    
    # --- 2. INDOT5 (ABSTRACTIVE) ---
    # Siapkan input untuk model
    input_text = "paraphrase: " + extractive_summary
    
    # Tokenisasi & Pindah ke GPU
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)
    
    # Generate (Inference)
    with torch.no_grad(): # Matikan gradien biar hemat VRAM & Cepat
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=256,
            min_length=30,
            num_beams=2,
            no_repeat_ngram_size=4,          # Beam search standar untuk kualitas
            repetition_penalty=1.5, # Mencegah pengulangan kata
            length_penalty=1.0,
            early_stopping=True,
            do_sample=False
        )
    
    abstractive_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return extractive_summary, abstractive_summary

print("✅ Fungsi Hybrid Pipeline Siap.")

✅ Fungsi Hybrid Pipeline Siap.


In [ ]:
from tqdm import tqdm
from tabulate import tabulate
import json

if "results" not in globals():
    results = []

if "errors" not in globals():
    errors = []

done_ids = set(r["doc_id"] for r in results)

print(f"🚀 Memulai / melanjutkan Hybrid Summarization...")
print(f"✅ Sudah selesai: {len(done_ids)} dokumen")
print(f"📄 Total data   : {len(df)} dokumen")

for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):

    doc_id = f"tempo_{index + 1:05d}"

    if doc_id in done_ids:
        continue

    try:
        source_text = str(row["step1_cleaned"])

        ext_sum, abs_sum = hybrid_summarization(source_text, top_n=3)

        results.append({
            "doc_id": doc_id,
            "No": row.get("No", index + 1),
            "Content": row.get("Content", ""),
            "Extractive_Summary": ext_sum,
            "Abstractive_Summary": abs_sum
        })

        done_ids.add(doc_id)

    except Exception as e:

        errors.append((doc_id, str(e)))

        results.append({
            "doc_id": doc_id,
            "No": row.get("No", index + 1),
            "Content": row.get("Content", ""),
            "Extractive_Summary": "ERROR",
            "Abstractive_Summary": "ERROR"
        })

        done_ids.add(doc_id)


print(f"\n✅ Selesai / berhenti di: {len(results)} dokumen")
print(f"❌ Error: {len(errors)}")


# =========================================================
# PRINT PREVIEW 5 DOKUMEN
# =========================================================

rows = [
    [
        r["doc_id"],
        r["No"],
        r["Content"],
        r["Extractive_Summary"],
        r["Abstractive_Summary"]
    ]
    for r in results[:5]
]

print("\n📄 Output Sistem: TextRank → Abstractive Summary\n")

print(
    tabulate(
        rows,
        headers=[
            "doc_id",
            "No",
            "Content",
            "gabungan kalimat textrank",
            "abstractive summary (T5)"
        ],
        tablefmt="fancy_grid",
        maxcolwidths=[14, 5, 45, 60, 60],
        showindex=False
    )
)


# =========================================================
# SAVE JSONL
# =========================================================

output_file = "tempo_hybrid_results_textrank_fix.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("\n🎉 File berhasil disimpan!")
print(f"📁 Output : {output_file}")
print(f"📊 Total  : {len(results)}")

🚀 Memulai / melanjutkan Hybrid Summarization...
✅ Sudah selesai: 0 dokumen
📄 Total data   : 300 dokumen


Processing:   0%|          | 0/300 [00:00<?, ?it/s]

Processing: 100%|██████████| 300/300 [07:05<00:00,  1.42s/it]


✅ Selesai / berhenti di: 300 dokumen
❌ Error: 0

📄 Output Sistem: TextRank → Abstractive Summary

╒═════════════╤══════╤═══════════════════════════════════════════════╤══════════════════════════════════════════════════════════════╤══════════════════════════════════════════════════════════════╕
│ doc_id      │   No │ Content                                       │ gabungan kalimat textrank                                    │ abstractive summary (T5)                                     │
╞═════════════╪══════╪═══════════════════════════════════════════════╪══════════════════════════════════════════════════════════════╪══════════════════════════════════════════════════════════════╡
│ tempo_00001 │    1 │ Indeks Harga Saham Gabungan (IHSG) Bursa Efek │ Indeks Harga Saham Gabungan (IHSG) Bursa Efek Indonesia      │ Dalam kajiannya di Jakarta, Rabu (16/7), Tim Riset Phintraco │
│             │      │ Indonesia (BEI) ditutup menguat pada akhir    │ (BEI) ditutup menguat pada akhir perdagang